In [1]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px
import gcamreader

In [2]:
def ej_to_twh(ej):
    """
    Convert energy from exajoules (EJ) to terawatt-hours (TWh).

    Parameters:
    ej (float): Energy in exajoules.

    Returns:
    float: Energy in terawatt-hours.
    """
    twh = ej * 277.777778
    return twh

In [3]:
# =========================
# Config
# =========================
PROJECT_PATH = Path("/data/project/tae/gcam-core")
DB_REL_PATH  = "../output"
DB_FILE      = "database_basexdb_korea_2035_v5"
QUERY_FILE   = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION       = "South Korea"
SCENARIOS    = ["High-Ambition-Med", "High-Ambition-Med-CPO2040"]
Q_GEN_TECH   = 9   # electricity generation by technology
YEARS_MARK   = list(range(2015, 2036, 5))  # for share markers/annotations

# Colors
custom_colors = {
    'Solar': '#FECB52',
    'Wind': 'rgb(136,204,238)',
    'Hydro': 'rgb(95, 70, 144)',
    'Nuclear': '#AB63FA',
    'Biomass': 'rgb(115, 175, 72)',
    'Gas w/ CCS': '#DEA0FD',
    'Gas': '#FFA15A',
    'Coal w/ CCS': '#750D86',
    'Coal': '#222A2A',
    'Oil': '#7D1215',
    'Hydrogen': "#727DCD",
    'Ammonia': "rgb(231,63,116)",
    'Others': 'rgb(217,217,217)',
}

stack_order = [
    'Ammonia', 'Hydrogen', 'Coal w/ CCS', 'Coal',
    'Gas w/ CCS', 'Gas', 'Oil', 'Nuclear',
    'Biomass', 'Hydro', 'Wind', 'Solar'
]

stack_order

['Ammonia',
 'Hydrogen',
 'Coal w/ CCS',
 'Coal',
 'Gas w/ CCS',
 'Gas',
 'Oil',
 'Nuclear',
 'Biomass',
 'Hydro',
 'Wind',
 'Solar']

In [4]:
# =========================
# Helpers
# =========================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def run_query(conn, q_idx):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    df = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def cat_tech(t):
    if t in ['PV', 'PV_storage', 'rooftoop_pv']: return 'Solar'
    if t in ['wind', 'wind_offshore', 'wind_storage']: return 'Wind'
    if t in ['hydro']: return 'Hydro'
    if t in ['Gen_III', 'Gen_II_LWR']: return 'Nuclear'
    if t in ['biomass (IGCC CCS)', 'biomass (conv CCS)']: return 'Biomass w/ CCS'
    if t in ['biomass (IGCC)', 'biomass (conv)']: return 'Biomass'
    if t in ['gas (CC CCS)']: return 'Gas w/ CCS'
    if t in ['gas (CC)', 'gas (steam/CT)']: return 'Gas'
    if t in ['coal (IGCC CCS)', 'coal (conv pul CCS)']: return 'Coal w/ CCS'
    if t in ['coal (IGCC)', 'coal (conv pul)']: return 'Coal'
    if t in ['refined liquids (CC CCS)']: return 'Oil w/ CCS'
    if t in ['refined liquids (CC)', 'refined liquids (steam/CT)']: return 'Oil'
    if t in ['gas (CC H2 blend 50%)']: return 'Hydrogen'
    if t in ['coal (conv pul ammonia blend 20%)']: return 'Ammonia'
    return 'Others'

def reallocate_h2_nh3(pivot: pd.DataFrame) -> pd.DataFrame:
    """
    Split Hydrogen & Ammonia to backing fuels (50% H2 → 50% Gas; 20% NH3 → 80% Coal)
    while preserving totals.
    """
    out = pivot.copy()
    # keep originals to compute remainders
    H_orig  = out.get('Hydrogen', pd.Series(0, index=out.index))
    NH3_orig= out.get('Ammonia',  pd.Series(0, index=out.index))

    out['Hydrogen'] = H_orig * 0.5
    out['Ammonia']  = NH3_orig * 0.2

    # Add the remaining to Gas / Coal
    out['Gas']  = out.get('Gas', 0)  + H_orig * 0.5
    out['Coal'] = out.get('Coal', 0) + NH3_orig * 0.8
    return out

def calc_shares(df_long: pd.DataFrame, scenario: str) -> tuple[pd.Series, pd.Series]:
    """Return RE% and Carbon-free% (RE + Nuclear) time series for a scenario."""
    is_re = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia',])
    is_cf = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia', 'Nuclear',])
    tot   = df_long[df_long['scenario'] == scenario].groupby('Year')['value'].sum()
    re    = df_long[(df_long['scenario'] == scenario) & is_re].groupby('Year')['value'].sum()
    cf    = df_long[(df_long['scenario'] == scenario) & is_cf].groupby('Year')['value'].sum()
    re_share = (re / tot * 100).reindex(YEARS_MARK)
    cf_share = (cf / tot * 100).reindex(YEARS_MARK)
    return re_share, cf_share

rank_map = {name: i for i, name in enumerate(stack_order)}
max_rank = len(stack_order) - 1

rank_map = {name: i for i, name in enumerate(stack_order)}
max_rank = len(stack_order) - 1

def add_stack_bars(fig, df_side: pd.DataFrame, col: int, show_legend: bool):
    cats = [c for c in stack_order if c in set(df_side["genTech"])]

    for c in cats:
        sub = df_side[df_side["genTech"] == c]
        fig.add_bar(
            name=c,
            x=sub["Year"],
            y=sub["value"],
            marker=dict(color=custom_colors.get(c)),
            showlegend=show_legend,
            legendrank=10 + (max_rank - rank_map[c]),  # ✅ legend 역순
            row=1, col=col, secondary_y=False
        )

In [5]:
conn = connect_db()

Database scenarios: High-Ambition-Med, Current-Policies-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-High, Current-Policies-High, High-Ambition-Med, Current-Policies-Med, Current-Policies-High, High-Ambition-Low, Current-Policies-Low, High-Ambition-Med-AI, Current-Policies-Med-AI, High-Ambition-Med-CPO2040, High-Ambition-Med, High-Ambition-Med


In [6]:
df = run_query(conn=conn, q_idx=16)
df[(df['fuel'] == 'elect_td_ind') & (df['Year'] == 2035)]

,Units,scenario,region,fuel,Year,value
30,1975$/GJ,High-Ambition-Med,South Korea,elect_td_ind,2035,12.3399
118,1975$/GJ,High-Ambition-Med-CPO2040,South Korea,elect_td_ind,2035,12.1270


In [11]:
12.1270 / 12.3399 * 100

98.27470238818792

In [8]:
# 1) Load generation by technology
df = run_query(conn, Q_GEN_TECH)
df['genTech'] = df['technology'].map(cat_tech)
df = df[~df['genTech'].isna()].copy()

# 2) Convert EJ to TWh
df['value'] = df['value'].apply(ej_to_twh)
df['Units'] = 'TWh'

In [9]:
dfFig = (
    df[(df['Year'] >= 2015) & (df['Year'] <= 2035)]
    .groupby(['scenario', 'Year', 'genTech'], observed=False)['value']
    .sum().reset_index()
)
dfFig

,scenario,Year,genTech,value
0,High-Ambition-Med,2015,Biomass,2.324597
1,High-Ambition-Med,2015,Coal,215.851667
2,High-Ambition-Med,2015,Gas,120.367295
3,High-Ambition-Med,2015,Hydro,2.642000
4,High-Ambition-Med,2015,Nuclear,164.662778
...,...,...,...,...
99,High-Ambition-Med-CPO2040,2035,Nuclear,236.111111
100,High-Ambition-Med-CPO2040,2035,Oil,12.550611
101,High-Ambition-Med-CPO2040,2035,Others,12.607111
102,High-Ambition-Med-CPO2040,2035,Solar,173.437832


In [10]:
dfFig.groupby(['Year', 'scenario'])['value'].sum()

Year  scenario                 
2015  High-Ambition-Med            521.740750
      High-Ambition-Med-CPO2040    521.740750
2020  High-Ambition-Med            537.753683
      High-Ambition-Med-CPO2040    537.753683
2025  High-Ambition-Med            593.229512
      High-Ambition-Med-CPO2040    593.229512
2030  High-Ambition-Med            689.021452
      High-Ambition-Med-CPO2040    699.455425
2035  High-Ambition-Med            770.485528
      High-Ambition-Med-CPO2040    819.723877
Name: value, dtype: float64

In [9]:
pivot = dfFig.pivot_table(index=['scenario', 'Year'], columns='genTech', values='value', fill_value=0)
pivot

genTech                           Biomass        Coal  Coal w/ CCS  \
scenario                  Year                                       
High-Ambition-Med         2015   2.324597  215.851667     0.000000   
                          2020   2.290144  196.944722     0.000000   
                          2025   3.047140  153.906945     2.744733   
                          2030   5.266783   83.570833     8.333333   
                          2035  11.705750    0.215630    17.500000   
High-Ambition-Med-CPO2040 2015   2.324597  215.851667     0.000000   
                          2020   2.290144  196.944722     0.000000   
                          2025   3.047140  153.906945     2.744733   
                          2030   4.905681  102.466389     8.333333   
                          2035   9.937028   51.290278    17.500000   

genTech                                Gas  Gas w/ CCS     Hydro   Hydrogen  \
scenario                  Year                                                
High-Ambition-Med         2015  120.367295    0.000000  2.642000   0.000000   
                          2020  132.446161    0.000000  4.722222   0.000000   
                          2025  141.202436    2.152547  3.611111   0.000000   
                          2030  128.474961    4.168361  3.611111   6.029028   
                          2035   81.009714    8.333333  3.611111  17.934167   
High-Ambition-Med-CPO2040 2015  120.367295    0.000000  2.642000   0.000000   
                          2020  132.446161    0.000000  4.722222   0.000000   
                          2025  141.202436    2.152547  3.611111   0.000000   
                          2030  123.524506    3.753528  3.611111   5.476528   
                          2035   78.736583    8.333333  3.611111  13.966833   

genTech                            Nuclear        Oil     Others       Solar  \
scenario                  Year                                                 
High-Ambition-Med         2015  164.662778  10.578611   0.000000    3.972611   
                          2020  160.277833  12.104056   0.568292   20.639963   
                          2025  180.917500  16.818611   5.195333   47.203665   
                          2030  204.166945  11.258917   5.991139  105.382276   
                          2035  236.111111  13.799250  15.601417  173.437832   
High-Ambition-Med-CPO2040 2015  164.662778  10.578611   0.000000    3.972611   
                          2020  160.277833  12.104056   0.568292   20.639963   
                          2025  180.917500  16.818611   5.195333   47.203665   
                          2030  204.166667  10.370028   5.440222  105.382276   
                          2035  236.111111  12.550611  12.607111  173.437832   

genTech                               Wind  
scenario                  Year              
High-Ambition-Med         2015    1.341192  
                          2020    7.760289  
                          2025   36.429489  
                          2030  123.943989  
                          2035  207.483017  
High-Ambition-Med-CPO2040 2015    1.341192  
                          2020    7.760289  
                          2025   36.429489  
                          2030  122.025156  
                          2035  201.642045

In [10]:
out = pivot.copy()
H_orig  = out.get('Hydrogen', pd.Series(0, index=out.index))
NH3_orig= out.get('Ammonia',  pd.Series(0, index=out.index))

out['Hydrogen'] = H_orig * 0.5
out['Ammonia']  = NH3_orig * 0.2

out['Gas']  = out.get('Gas', 0)  + H_orig * 0.5
out['Coal'] = out.get('Coal', 0) + NH3_orig * 0.8

In [11]:
result_df = (
    out
    .reset_index()
    .melt(id_vars=['scenario', 'Year'], var_name='genTech', value_name='value')
)

# Order & clean
result_df['genTech'] = pd.Categorical(result_df['genTech'], categories=stack_order, ordered=True)
result_df = result_df.sort_values(['scenario', 'Year', 'genTech'])

# 4) Shares
re_current, cf_current = calc_shares(result_df, SCENARIOS[0])
re_enh,    cf_enh      = calc_shares(result_df, SCENARIOS[1])

In [14]:
fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=True, shared_yaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("High Ambition", "HA - CPO2040"),
    column_widths=[0.5, 0.5],
    horizontal_spacing=0.08
)

# ----------------------------
# Bars
# ----------------------------
left  = result_df[result_df['scenario'] == SCENARIOS[0]]
right = result_df[result_df['scenario'] == SCENARIOS[1]]

add_stack_bars(fig, left,  col=1, show_legend=True)
add_stack_bars(fig, right, col=2, show_legend=False)

# ----------------------------
# Share markers (REAL) — legend off
# ----------------------------
fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=re_current, mode="markers",
    name="Renewable Share",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=False
), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=cf_current, mode="markers",
    name="Carbon-Free Share",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    showlegend=False
), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=re_enh, mode="markers",
    name="Renewable Share",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=False
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=cf_enh, mode="markers",
    name="Carbon-Free Share",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    showlegend=False
), row=1, col=2, secondary_y=True)

# ----------------------------
# Annotate % values near markers
# ----------------------------
for year in YEARS_MARK:
    if pd.notna(re_current.get(year)):
        fig.add_annotation(
            x=year, y=re_current.get(year) + 3, text=f"{re_current.get(year):.0f}%",
            xref="x1", yref="y2", showarrow=False, font=dict(color="green")
        )
    if pd.notna(cf_current.get(year)):
        fig.add_annotation(
            x=year, y=cf_current.get(year) + 3, text=f"{cf_current.get(year):.0f}%",
            xref="x1", yref="y2", showarrow=False, font=dict(color="blue")
        )
    if pd.notna(re_enh.get(year)):
        fig.add_annotation(
            x=year, y=re_enh.get(year) + 3, text=f"{re_enh.get(year):.0f}%",
            xref="x2", yref="y4", showarrow=False, font=dict(color="green")
        )
    if pd.notna(cf_enh.get(year)):
        fig.add_annotation(
            x=year, y=cf_enh.get(year) + 3, text=f"{cf_enh.get(year):.0f}%",
            xref="x2", yref="y4", showarrow=False, font=dict(color="blue")
        )

# ----------------------------
# Dummy legend entries (Share only) + spacing line
#   - legendrank를 크게 줘서 발전원 아래로 보내기
#   - 공백 trace 하나로 간격 만들기
# ----------------------------
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(size=0, opacity=0),
    name=" ",  # spacing line
    showlegend=True,
    hoverinfo="skip",
    legendrank=1999
))



fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    name="Carbon-Free Share",
    showlegend=True,
    legendrank=2001
))


fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    name="Renewable Share",
    showlegend=True,
    legendrank=2003
))

# ----------------------------
# Layout / axes
# ----------------------------
fig.update_layout(
    width=850,
    height=700,
    barmode="stack",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(
        traceorder="normal",   # legendrank 적용 정렬
        x=1.02, y=1,
        font=dict(size=15),
        borderwidth=0
    ),
    margin=dict(r=220)  # legend 오른쪽 공간 확보 (필요시 조정)
)

# Primary y (left title only)
fig.update_yaxes(
    title_text="Electricity Generation (TWh)",
    row=1, col=1, secondary_y=False,
    showgrid=True, gridcolor="lightgray"
)
fig.update_yaxes(
    title_text=None,
    row=1, col=2, secondary_y=False,
    showgrid=True, gridcolor="lightgray"
)

# Secondary y-axes (share %): hide ticks, no grid, fixed range
fig.update_yaxes(
    secondary_y=True, range=[0, 100],
    showticklabels=False, title_text=None,
    showgrid=False,
    row=1, col=1
)
fig.update_yaxes(
    secondary_y=True, range=[0, 100],
    showticklabels=False, title_text=None,
    showgrid=False,
    row=1, col=2
)

# X axes
for c in (1, 2):
    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(2015, 2040, 5)),
        tickangle=45,
        tickfont=dict(size=15),
        row=1, col=c
    )

fig.update_layout(
    legend=dict(
        traceorder="normal",  # ✅ legendrank 정렬 유지
        font=dict(size=15),
        x=1.02, y=1,
        borderwidth=0
    )
)
# Fonts
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_annotations(font=dict(size=21))

pio.write_image(fig, "./fig/power_cpo2040.jpg", width=850, height=700, scale=3)
fig.show()


In [30]:
result_df

,scenario,Year,genTech,value
0,Current-Policies-Med,2015,Ammonia,0.000000
70,Current-Policies-Med,2015,Hydrogen,0.000000
30,Current-Policies-Med,2015,Coal w/ CCS,0.000000
20,Current-Policies-Med,2015,Coal,215.851667
50,Current-Policies-Med,2015,Gas w/ CCS,0.000000
...,...,...,...,...
19,High-Ambition-Med,2035,Biomass,11.705750
69,High-Ambition-Med,2035,Hydro,3.611111
129,High-Ambition-Med,2035,Wind,207.483017
119,High-Ambition-Med,2035,Solar,173.437832


In [31]:
def calc_shares(df_long: pd.DataFrame, scenario: str) -> tuple[pd.Series, pd.Series]:
    """Return RE% and Carbon-free% (RE + Nuclear) time series for a scenario."""
    is_re = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia', 'Others'])
    is_cf = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Hydrogen', 'Ammonia', 'Nuclear', 'Others'])
    tot   = df_long[df_long['scenario'] == scenario].groupby('Year')['value'].sum()
    re    = df_long[(df_long['scenario'] == scenario) & is_re].groupby('Year')['value'].sum()
    cf    = df_long[(df_long['scenario'] == scenario) & is_cf].groupby('Year')['value'].sum()
    re_share = (re / tot * 100).reindex(YEARS_MARK)
    cf_share = (cf / tot * 100).reindex(YEARS_MARK)
    return re_share, cf_share

In [32]:
df['genTech'] = df['technology'].apply(cat_tech)
df[(df['Year'] == 2030)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
1,TWh,Current-Policies-Med,South Korea,biomass,biomass (IGCC),electricity,2030,0.607097,Biomass
8,TWh,Current-Policies-Med,South Korea,biomass,biomass (conv),electricity,2030,3.738472,Biomass
11,TWh,Current-Policies-Med,South Korea,coal,coal (IGCC CCS),electricity,2030,1.243936,Coal w/ CCS
14,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul CCS),electricity,2030,0.772628,Coal w/ CCS
16,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul ammonia blend 20%),electricity,2030,41.111111,Ammonia
24,TWh,Current-Policies-Med,South Korea,coal,coal (conv pul),electricity,2030,77.366111,Coal
27,TWh,Current-Policies-Med,South Korea,gas,gas (CC CCS),electricity,2030,1.313808,Gas w/ CCS
29,TWh,Current-Policies-Med,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,9.291861,Hydrogen
37,TWh,Current-Policies-Med,South Korea,gas,gas (CC),electricity,2030,124.706111,Gas
45,TWh,Current-Policies-Med,South Korea,gas,gas (steam/CT),electricity,2030,0.359447,Gas
